# Bank Account Fraud Detection: Fraud Detection with Machine Learning and Deep Learning

**Dataset:** Bank Account Fraud (BAF) suite, NeurIPS 2022, file `Base.csv` (Kaggle `sgpjesus/bank-account-fraud-dataset-neurips-2022`) | **Tools:** Python, scikit-learn, LightGBM, Keras | **Platform:** Google Colab

**Problem statement.** When someone applies online for a bank account, the bank must decide in seconds whether the application is genuine or a **fraudulent account opening** (stolen identity, fake identity, money-mule account). Only about 1 in 100 applications is fraud, so checking everything by hand is impossible. The bank needs a model that gives every application a **fraud risk score** and sends the riskiest ones to an analyst first.

**Business objective.** Catch as much fraud as possible without flagging too many genuine customers (every false alarm delays a real customer and costs analyst time).

**ML objective.** Predict `fraud_bool` (1 = fraud, 0 = genuine) from the application details, the device and session behaviour and the recent "velocity" counts. Compare Logistic Regression, Random Forest and a tuned LightGBM model.

**DL objective.** Train a small Keras neural network (MLP) on the same data and check honestly whether it beats the machine learning models.

**Dataset.** 1,000,000 applications (32 columns) collected over **8 months** (`month` = 0 to 7). The file stores "missing" as `-1`, not as an empty cell, so we handle that first. `month` is used **only to split the data by time**, never as a model input. **No number in this notebook is typed in by hand: every count, rate and metric is calculated when you run the cells.**

**How to run.** Runtime > Run all. A full run should take roughly 20 to 40 minutes on a free Colab CPU. Set `FAST_MODE = True` in the next cell for a quick test (smaller sample, fewer trees, fewer epochs).

In [ ]:
!pip install -q imbalanced-learn shap kagglehub

import os, glob, gc, time, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
from imblearn.under_sampling import RandomUnderSampler      # resampling is only ever applied to TRAINING data
from imblearn.over_sampling import RandomOverSampler, SMOTE
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

warnings.filterwarnings("ignore")
SEED = 42
FAST_MODE = False                          # True = quick test run
SAMPLE_FRAC = 0.10 if FAST_MODE else 0.30  # share of the 1M rows used (1.0 = everything, needs more RAM and time)
N_TREES, N_ITER, EPOCHS = (60, 2, 3) if FAST_MODE else (300, 8, 40)
COST_FN, COST_FP = 500, 10                 # ASSUMED costs: a missed fraud costs 500, a false alarm costs 10 (edit to your case)
TARGET = "fraud_bool"
TRAIN_MONTHS, VAL_MONTHS, TEST_MONTHS = range(0, 5), [5], [6, 7]   # train on the past, tune on month 5, test on the latest months
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid"); plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 60); pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
BLUE, RED = "#4C72B0", "#C44E52"; PALETTE = {0: BLUE, 1: RED}

def interpret(*lines):
    """Short data-driven interpretation shown under an output."""
    display(Markdown("**Interpretation**\n\n" + "\n".join(f"- {l}" for l in lines)))

---
# Phase 1: Data Understanding, Cleaning and EDA

We first load the data and check what is really in the file, because nothing is assumed. We then clean it, and explore it with charts to see what fraudulent applications look like.

In [ ]:
def find_base_csv():
    """Download with kagglehub; if Kaggle cannot be reached, ask for a manual upload of Base.csv."""
    try:
        import kagglehub
        folder = kagglehub.dataset_download("sgpjesus/bank-account-fraud-dataset-neurips-2022")
        hits = glob.glob(os.path.join(folder, "**", "Base.csv"), recursive=True)
        if hits: return hits[0]
    except Exception as err:
        print("kagglehub failed:", repr(err))
    from google.colab import files
    return [k for k in files.upload() if k.lower().endswith(".csv")][0]

raw_df = pd.read_csv(find_base_csv())
N_RAW = len(raw_df)
df = raw_df.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True) if SAMPLE_FRAC < 1 else raw_df.copy()
del raw_df; gc.collect()
print(f"Full file: {N_RAW:,} rows | working sample: {len(df):,} rows x {df.shape[1]} columns")
display(df.head(3)); df.info()

# Decide the role of every column from the real file
assert TARGET in df.columns and "month" in df.columns, f"Expected columns missing. Columns: {list(df.columns)}"
SENTINEL   = [c for c in ["prev_address_months_count", "current_address_months_count", "bank_months_count",
                          "session_length_in_minutes", "device_distinct_emails_8w"] if c in df.columns]    # -1 means "missing"
CAT_RAW    = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
NUM_RAW    = [c for c in df.columns if c not in CAT_RAW + [TARGET, "month"]]
BINARY_RAW = [c for c in NUM_RAW if df[c].nunique() == 2]
interpret(f"Target: **{TARGET}**. Categories: {CAT_RAW}. Binary flags: {BINARY_RAW}. Time column: **month** ({int(df['month'].min())} to {int(df['month'].max())}), used only for splitting.",
          f"Columns where `-1` means 'missing': {SENTINEL} (plus negative values in `intended_balcon_amount`). There is no calendar date, so we split by `month`.")

In [ ]:
# ---- Data quality checks ----
HIDDEN = SENTINEL + [c for c in ["intended_balcon_amount"] if c in df.columns]
def is_missing(frame, c):
    """BAF stores 'missing' as -1 (or a negative number in intended_balcon_amount), not as NaN."""
    return frame[c] < 0 if c == "intended_balcon_amount" else frame[c] == -1

miss = pd.DataFrame({"hidden_missing_pct": {c: is_missing(df, c).mean() * 100 for c in HIDDEN}}).sort_values("hidden_missing_pct", ascending=False)
display(miss.round(2))
n_nan, n_dup = int(df.isna().sum().sum()), int(df.duplicated().sum())
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
checks = {"income outside [0, 1]": ((df["income"] < 0) | (df["income"] > 1)).sum(),
          "name_email_similarity outside [0, 1]": ((df["name_email_similarity"] < 0) | (df["name_email_similarity"] > 1)).sum(),
          "negative days_since_request": (df["days_since_request"] < 0).sum(),
          "negative velocity values": (df[["velocity_6h", "velocity_24h", "velocity_4w"]] < 0).any(axis=1).sum()}

# ---- Cleaning ----
df_model = df.drop_duplicates().reset_index(drop=True)             # duplicates in both train and test would leak
df_plot = df_model.copy()                                          # plotting copy: hidden missing values become real NaN
for c in HIDDEN: df_plot[c] = df_plot[c].where(~is_missing(df_plot, c))

interpret(f"Most missing column: **{miss.index[0]}** ({miss['hidden_missing_pct'].iloc[0]:.1f}% of rows hold `-1` or a negative value). A scaler or model would read `-1` as a real number, so we convert it to NaN and keep an explicit missing flag.",
          f"Normal NaN cells: {n_nan}. Duplicate rows removed: {n_dup}. Constant columns (no information, dropped later): {const_cols or 'none'}.",
          f"Impossible values: {({k: int(v) for k, v in checks.items()})}.",
          f"Clean table: **{len(df_model):,} rows**. Outliers (burst of applications, huge counts) are **kept**, because in fraud the extreme values are often the signal.")

### 1.1 Class imbalance and drift over time
Fraud is rare. If only a small share of applications are fraud, a model that always says "genuine" looks very accurate and catches nothing. We measure the imbalance first, and also check whether the fraud rate changes from month to month (this decides how we split the data).

In [ ]:
counts = df_model[TARGET].value_counts().sort_index()
n_pos_all, n_neg_all = int(counts.get(1, 0)), int(counts.get(0, 0))
fraud_rate = df_model[TARGET].mean(); imbalance_ratio = n_neg_all / max(n_pos_all, 1)
by_month = df_model.groupby("month")[TARGET].agg(rate="mean", n_fraud="sum", n_apps="size"); by_month["rate"] *= 100

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.barplot(x=["Genuine (0)", "Fraud (1)"], y=counts.values, palette=[BLUE, RED], ax=ax[0])
for i, v in enumerate(counts.values): ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
ax[0].set_title("Applications by class")
ax[1].pie(counts.values, labels=["Genuine", "Fraud"], colors=[BLUE, RED], autopct="%1.2f%%", startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"))
ax[1].set_title("Class share")
ax[2].plot(by_month.index, by_month["rate"], marker="o", color=RED); ax[2].set(title="Fraud rate (%) by month", xlabel="month", ylabel="fraud rate %")
plt.tight_layout(); plt.show()
interpret(f"{n_neg_all:,} genuine vs {n_pos_all:,} fraud applications: fraud is **{fraud_rate * 100:.2f}%** of the data (about {imbalance_ratio:.0f} genuine for every fraud).",
          f"Always predicting 'genuine' would be {100 - fraud_rate * 100:.2f}% accurate and catch **zero** fraud, so we judge models with **recall, precision, F1, ROC-AUC and PR-AUC**, not accuracy.",
          f"Fraud rate by month goes from {by_month['rate'].min():.2f}% to {by_month['rate'].max():.2f}%. Fraud behaviour changes over time, so we train on the past and test on the future (a time split) instead of shuffling randomly.")

### 1.2 Missing fields and flags as fraud signals
Genuine customers usually have a history (previous address, previous bank account, device history). If that history is **missing**, the application may be riskier. We compare the fraud rate when a field is missing with when it is present, and the fraud rate for each 0/1 flag.

In [ ]:
rows = []
for c in HIDDEN:
    m = is_missing(df_model, c)
    if 0 < m.sum() < len(m): rows.append((c, m.mean() * 100, df_model.loc[m, TARGET].mean() * 100, df_model.loc[~m, TARGET].mean() * 100))
gap = pd.DataFrame(rows, columns=["column", "pct_missing", "fraud_if_missing", "fraud_if_present"])
gap["gap"] = gap["fraud_if_missing"] - gap["fraud_if_present"]; gap = gap.sort_values("gap", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
g = gap.melt(id_vars="column", value_vars=["fraud_if_missing", "fraud_if_present"], var_name="state", value_name="fraud_rate")
sns.barplot(data=g, y="column", x="fraud_rate", hue="state", palette=[RED, BLUE], ax=ax[0]); ax[0].axvline(fraud_rate * 100, ls="--", c="grey")
ax[0].set_title("Fraud rate (%) when a field is missing vs present"); ax[0].set_ylabel("")
fl = pd.DataFrame([(f"{c} = {int(v)}", r * 100) for c in BINARY_RAW for v, r in df_model.groupby(c)[TARGET].mean().items()], columns=["label", "rate"])
sns.barplot(data=fl, y="label", x="rate", color=RED, ax=ax[1]); ax[1].axvline(fraud_rate * 100, ls="--", c="grey")
ax[1].set_title("Fraud rate (%) by flag value"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()
top = gap.iloc[0]; fl_top = fl.sort_values("rate", ascending=False).iloc[0]
interpret(f"Biggest gap: **{top['column']}** has fraud rate {top['fraud_if_missing']:.2f}% when missing vs {top['fraud_if_present']:.2f}% when present (dashed line = overall {fraud_rate * 100:.2f}%).",
          f"Highest-risk flag value: **{fl_top['label']}** with fraud rate {fl_top['rate']:.2f}%.",
          "Where the red 'missing' bar is far above the blue one, *being missing* is itself a signal, so we create a missing flag for each of these columns in feature engineering.")

### 1.3 Categories and age
We check which category values and which age groups have more fraud (only groups with at least 200 applications, since tiny groups give unreliable rates).

In [ ]:
cat_show = CAT_RAW[:5]
fig, axes = plt.subplots(2, 3, figsize=(18, 9)); axes = axes.ravel()
best_cat = []
for ax, c in zip(axes, cat_show):
    t = df_model.groupby(c)[TARGET].agg(count="size", rate="mean").reset_index()
    t = t[t["count"] >= 200].sort_values("rate", ascending=False); t["rate"] *= 100
    sns.barplot(data=t, y=c, x="rate", color=RED, ax=ax); ax.axvline(fraud_rate * 100, ls="--", c="grey"); ax.set_title(f"Fraud rate (%) by {c}"); ax.set_ylabel("")
    if len(t): best_cat.append(f"`{c}`: **{t.iloc[0][c]}** at {t.iloc[0]['rate']:.1f}% (n={int(t.iloc[0]['count']):,})")
age_t = df_model.groupby("customer_age")[TARGET].mean() * 100
sns.barplot(x=age_t.index.astype(int).astype(str), y=age_t.values, color=RED, ax=axes[len(cat_show)])
axes[len(cat_show)].axhline(fraud_rate * 100, ls="--", c="grey"); axes[len(cat_show)].set_title("Fraud rate (%) by customer_age bucket")
for ax in axes[len(cat_show) + 1:]: ax.axis("off")
plt.tight_layout(); plt.show()
interpret("Highest fraud rate among frequent groups: " + "; ".join(best_cat) + ".",
          f"Age bucket with the highest fraud rate: **{int(age_t.idxmax())}** ({age_t.max():.2f}%), lowest: **{int(age_t.idxmin())}** ({age_t.min():.2f}%).",
          "Age is a sensitive attribute. A model may use it as a shortcut and treat older or younger genuine customers unfairly, so we run a fairness check in Phase 7.")

### 1.4 Correlation
Correlation shows linear links only, so a weak value does not make a feature useless. In fraud data the signal is usually spread over many weak, non-linear patterns. The second chart shows features that say almost the same thing (redundant).

In [ ]:
corr = df_plot[NUM_RAW + [TARGET]].corr()
tcorr = corr[TARGET].drop(TARGET).dropna().sort_values(key=np.abs, ascending=False)
top12 = tcorr.head(12).index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(17, 6))
t15 = tcorr.head(15)[::-1]; ax[0].barh(t15.index, t15.values, color=[RED if v > 0 else BLUE for v in t15.values]); ax[0].set_title("Correlation with fraud_bool (top 15)")
sns.heatmap(corr.loc[top12, top12], cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, cbar=False, ax=ax[1]); ax[1].set_title("Correlation between the 12 strongest features")
plt.tight_layout(); plt.show()
cm = corr.drop(index=TARGET, columns=TARGET).abs()
pairs = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool)).stack(); pairs = pairs[pairs > 0.7].sort_values(ascending=False)
interpret(f"Strongest linear link with the target: **{tcorr.index[0]}** ({tcorr.iloc[0]:.3f}). Values this small are normal in fraud data.",
          "Redundant pairs (|r| > 0.7): " + (", ".join(f"{a} and {b} ({v:.2f})" for (a, b), v in pairs.head(3).items()) if len(pairs) else "none") + ". This hurts Logistic Regression a little but not tree models.")

---
# Phase 2: Feature Engineering, Preprocessing and Split

**Leakage rule.** Every engineered feature is calculated **row by row from that application's own fields**, with no use of the target and no statistics from other rows. Anything that has to *learn* from data (median for filling gaps, scaling, one-hot categories) is fitted on the **training months only**.

| Feature | Why it can help |
|---|---|
| `<column>_missing` flags, `n_missing_flags` | Missing history can itself signal fraud |
| `credit_limit_to_income` | A very high credit limit compared with income is risky |
| `velocity_6h_to_24h`, `velocity_24h_to_4w` | Shows a sudden burst of applications (acceleration) |
| `address_months_total`, `address_to_age` | Living-stability of the applicant |
| `phones_valid` | How many phone numbers could be verified |
| `low_similarity_free_email` | Free email address that does not look like the applicant's name |

**Why a time split.** A random split would mix months, so the model would be tested on the same period it learned from. Fraud changes over time, so we train on months 0 to 4, tune on month 5 and test once on months 6 and 7, like a real bank would (learn from the past, face the future). `month` itself is never a feature.

In [ ]:
FLAG_COLS = [c + "_missing" for c in HIDDEN]
NEW_FEATS = ["credit_limit_to_income", "velocity_6h_to_24h", "velocity_24h_to_4w", "address_months_total", "address_to_age",
             "phones_valid", "low_similarity_free_email", "n_missing_flags"] + FLAG_COLS

def prepare(d):
    """Row-by-row cleaning and features only, so it is safe to run before the split."""
    d = d.copy()
    for c in HIDDEN:
        miss_c = is_missing(d, c)
        d[c + "_missing"] = miss_c.astype("int8")                     # flag BEFORE the value is blanked
        d[c] = d[c].where(~miss_c)                                    # -1 becomes a real NaN
    d["credit_limit_to_income"]   = d["proposed_credit_limit"] / (d["income"] + 1e-3)
    d["velocity_6h_to_24h"]       = d["velocity_6h"] / (d["velocity_24h"].abs() + 1e-3)
    d["velocity_24h_to_4w"]       = d["velocity_24h"] / (d["velocity_4w"].abs() + 1e-3)
    d["address_months_total"]     = d["current_address_months_count"].fillna(0) + d["prev_address_months_count"].fillna(0)
    d["address_to_age"]           = d["current_address_months_count"] / (d["customer_age"] * 12)
    d["phones_valid"]             = d["phone_home_valid"] + d["phone_mobile_valid"]
    d["low_similarity_free_email"] = ((d["name_email_similarity"] < 0.2) & (d["email_is_free"] == 1)).astype("int8")
    d["n_missing_flags"]          = d[FLAG_COLS].sum(axis=1)
    return d.drop(columns=[c for c in const_cols if c in d.columns])  # constant columns help nobody

df_p = prepare(df_model)
FEATURES = [c for c in df_p.columns if c not in [TARGET, "month"]]
CAT_FEATURES = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df_p[c])]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

# Do the new features carry signal? (how well one feature alone ranks fraud above genuine: 0.5 = no signal)
sig = pd.Series({c: roc_auc_score(df_p[TARGET], df_p[c].fillna(df_p[c].median())) for c in NEW_FEATS if df_p[c].nunique() > 1}, name="single_feature_AUC")
sig = sig.reindex((sig - 0.5).abs().sort_values(ascending=False).index)
display(sig.head(8).to_frame())
interpret(f"{len(FEATURES)} candidate features: {len(NUM_FEATURES)} numeric (including flags) and {len(CAT_FEATURES)} categorical. `{TARGET}` and `month` are excluded.",
          f"Strongest new feature on its own: **{sig.index[0]}** (AUC {sig.iloc[0]:.3f}; 0.5 would mean no signal, below 0.5 means it points the other way). Whether the new features really help is checked later with feature importance.")

In [ ]:
def make_preprocessor():
    """Fresh, unfitted preprocessing: fill gaps with the median and scale numbers, one-hot encode categories."""
    return ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_FEATURES),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES)])

tr, va, te = [df_p["month"].isin(m) for m in (TRAIN_MONTHS, VAL_MONTHS, TEST_MONTHS)]
y_train, y_val, y_test = [df_p.loc[m, TARGET].values.astype(int) for m in (tr, va, te)]
prep = make_preprocessor()
X_train = prep.fit_transform(df_p.loc[tr, FEATURES]).astype("float32")           # fitted on TRAIN months only
X_val, X_test = [prep.transform(df_p.loc[m, FEATURES]).astype("float32") for m in (va, te)]
FEAT_NAMES = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum()); SPW = n_neg / n_pos   # genuine applications per fraud in TRAIN

split_tbl = pd.DataFrame({"Rows": [len(y_train), len(y_val), len(y_test)], "Fraud": [y_train.sum(), y_val.sum(), y_test.sum()],
                          "Genuine": [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
                          "Fraud rate %": [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100]},
                         index=["Train (months 0-4)", "Validation (month 5)", "Test (months 6-7)"])
display(split_tbl.round(3))
BASELINE_ACC = accuracy_score(y_test, np.zeros_like(y_test))
interpret(f"Train: {len(y_train):,} applications ({n_pos:,} fraud). Validation: {len(y_val):,} ({int(y_val.sum())} fraud). Test: {len(y_test):,} ({int(y_test.sum())} fraud). With so few fraud cases in the test months, one extra miss moves recall by about {100 / max(int(y_test.sum()), 1):.2f} points, so read small differences with care.",
          f"The 'always genuine' baseline reaches {BASELINE_ACC * 100:.2f}% accuracy with recall 0. After preprocessing (fitted on train only) the model sees {X_train.shape[1]} columns.",
          "Validation is used to tune models and to choose thresholds. The test months are used only for the final scores.")

In [ ]:
# ---- Evaluation helpers used for the rest of the notebook ----
def tpr_at_fpr(y, p, target=0.05):
    """Share of fraud caught if at most `target` (5%) of genuine applicants may be flagged."""
    fpr, tpr, _ = roc_curve(y, p); return float(tpr[fpr <= target].max())
def thr_at_fpr(y, p, target=0.05):
    fpr, tpr, thr = roc_curve(y, p); return float(thr[np.where(fpr <= target)[0][-1]])
def thr_best_f1(y, p):
    pr, rc, thr = precision_recall_curve(y, p); f1 = 2 * pr * rc / (pr + rc + 1e-12); return float(thr[np.nanargmax(f1[:-1])])
def confusion(y, prob, thr):
    tn, fp, fn, tp = confusion_matrix(y, (np.asarray(prob) >= thr).astype(int), labels=[0, 1]).ravel(); return tn, fp, fn, tp
def compute_metrics(y, prob, thr):
    pred = (np.asarray(prob) >= thr).astype(int); tn, fp, fn, tp = confusion(y, prob, thr)
    return {"Accuracy": accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0), "ROC_AUC": roc_auc_score(y, prob), "PR_AUC": average_precision_score(y, prob),
            "TPR@5%FPR": tpr_at_fpr(y, prob), "TP": tp, "FP": fp, "FN": fn, "TN": tn}

RESULTS = {}
def add_result(name, kind, val_prob, test_prob, est=None):
    """Store scores and test metrics. The 0/1 threshold is chosen on the VALIDATION month (max F1), never on the test months."""
    thr = thr_best_f1(y_val, val_prob)
    m = compute_metrics(y_test, test_prob, thr); m["Threshold"] = thr
    RESULTS[name] = {"kind": kind, "val": np.asarray(val_prob).ravel(), "test": np.asarray(test_prob).ravel(), "metrics": m, "estimator": est}
    print(f"{name:<20} Prec={m['Precision']:.4f} Rec={m['Recall']:.4f} F1={m['F1']:.4f} ROC-AUC={m['ROC_AUC']:.4f} PR-AUC={m['PR_AUC']:.4f} TPR@5%FPR={m['TPR@5%FPR']:.4f} | TP={m['TP']} FP={m['FP']} FN={m['FN']}")

---
# Phase 3: Handling Class Imbalance

We compare five ways to deal with the imbalance using the same fast LightGBM model: no handling, **class weights**, random undersampling, random oversampling and **SMOTE** (synthetic fraud examples). Every resampling step is fitted on the **training data only**, never on validation or test. The comparison is scored on the **validation month**, so the test months stay untouched.

In [ ]:
def quick_lgbm(w=1.0):
    return LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, scale_pos_weight=w, n_jobs=-1, random_state=SEED, verbose=-1)
strategies = {"No handling": (None, 1.0), "Class weight": (None, SPW),
              "Undersampling (1:10)": (RandomUnderSampler(sampling_strategy=0.1, random_state=SEED), 1.0),
              "Oversampling (1:10)": (RandomOverSampler(sampling_strategy=0.1, random_state=SEED), 1.0),
              "SMOTE (1:10)": (SMOTE(sampling_strategy=0.1, random_state=SEED), 1.0)}
rows = []
for name, (sampler, w) in strategies.items():
    Xr, yr = (X_train, y_train) if sampler is None else sampler.fit_resample(X_train, y_train)      # TRAIN ONLY
    pv = quick_lgbm(w).fit(Xr, yr).predict_proba(X_val)[:, 1]
    rows.append({"Strategy": name, "train_rows": len(yr), "PR_AUC": average_precision_score(y_val, pv), "ROC_AUC": roc_auc_score(y_val, pv), "TPR@5%FPR": tpr_at_fpr(y_val, pv)})
imb = pd.DataFrame(rows).set_index("Strategy"); display(imb.round(4))
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, (col, color) in zip(ax, [("train_rows", "grey"), ("PR_AUC", "teal"), ("TPR@5%FPR", RED)]): imb[col].plot.barh(ax=a, color=color); a.set_title(col + " (validation)" if col != "train_rows" else "Training rows after resampling"); a.set_ylabel("")
plt.tight_layout(); plt.show()
best_imb = imb["PR_AUC"].idxmax(); gap_cw = imb["PR_AUC"].max() - imb.loc["Class weight", "PR_AUC"]
imb_note = ("class weights had the best validation PR-AUC" if gap_cw <= 1e-12 else
            f"'{best_imb}' was higher than class weights by {gap_cw:.4f} PR-AUC" + (" (within noise, so the simpler class weights were kept)" if gap_cw < 0.01 else ", but class weights were kept because they leave the data untouched and are simple to explain"))
interpret(f"Best validation PR-AUC: **{best_imb}** ({imb['PR_AUC'].max():.4f}). Class weights reach {imb.loc['Class weight', 'PR_AUC']:.4f}; SMOTE reaches {imb.loc['SMOTE (1:10)', 'PR_AUC']:.4f}; no handling reaches {imb.loc['No handling', 'PR_AUC']:.4f}.",
          f"Decision: the main models use **class weights** (`class_weight` / `scale_pos_weight`). Note: {imb_note}.",
          "Undersampling throws most genuine rows away, and oversampling or SMOTE adds copies or synthetic points, which can look unrealistic. Class weights only tell the model that a missed fraud matters more.")

---
# Phase 4: Machine Learning Track

We compare three models, all trained on the training months only:

1. **Logistic Regression**, the simple, explainable baseline.
2. **Random Forest**, many decision trees voting together.
3. **LightGBM**, gradient boosting (each new tree fixes the errors of the earlier ones), very strong on tables like this one. We then **tune** it with `RandomizedSearchCV`, optimising **PR-AUC** (not accuracy), and compare it before and after tuning.

Every model is scored on the same untouched test months. The 0/1 threshold of each model is picked on the validation month (the value that gives the best F1). Section 6 then looks at better thresholds.

In [ ]:
models = {"Logistic Regression": LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
          "Random Forest": RandomForestClassifier(n_estimators=N_TREES // 2, max_depth=14, min_samples_leaf=10, max_features="sqrt",
                                                  class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
          "LightGBM": LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
                                     colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=SPW, n_jobs=-1, random_state=SEED, verbose=-1)}
for name, est in models.items():
    est.fit(X_train, y_train)
    add_result(name, "ML", est.predict_proba(X_val)[:, 1], est.predict_proba(X_test)[:, 1], est)
b = pd.DataFrame({n: RESULTS[n]["metrics"] for n in RESULTS}).T.astype(float)
interpret(f"Accuracy is not shown here on purpose: the 'always genuine' baseline already gets {BASELINE_ACC:.3f}. Recall ranges from {b['Recall'].min():.3f} to {b['Recall'].max():.3f} and PR-AUC from {b['PR_AUC'].min():.3f} to {b['PR_AUC'].max():.3f} (a model with no skill would get a PR-AUC of about {y_test.mean():.3f}).",
          f"Best baseline by PR-AUC: **{b['PR_AUC'].idxmax()}**. Precision is low for every model because even a small false-alarm rate on a huge genuine majority creates many alerts. This is normal for rare-event detection.")

In [ ]:
# Tune LightGBM: random search where every candidate trains on the TRAINING months and is scored on the VALIDATION month (no shuffled folds, no peeking at the future)
X_tv, y_tv = np.vstack([X_train, X_val]), np.concatenate([y_train, y_val])
cv_split = PredefinedSplit(np.r_[np.full(len(y_train), -1), np.zeros(len(y_val), dtype=int)])
space = {"n_estimators": [200, 300, 450], "num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1], "min_child_samples": [20, 50, 100],
         "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.5, 0.7, 0.9], "reg_lambda": [0, 5, 20], "scale_pos_weight": [1.0, float(np.sqrt(SPW)), float(SPW)]}
search = RandomizedSearchCV(LGBMClassifier(subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1), space, n_iter=N_ITER, cv=cv_split, random_state=SEED, n_jobs=1,
                            scoring={"pr_auc": "average_precision", "recall": "recall", "precision": "precision", "f1": "f1"}, refit=False)
t0 = time.time(); search.fit(X_tv, y_tv)
bi = int(np.argmin(search.cv_results_["rank_test_pr_auc"])); best_params = search.cv_results_["params"][bi]
tuned = LGBMClassifier(subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1).set_params(**best_params).fit(X_train, y_train)   # re-fit on train months only
add_result("LightGBM (Tuned)", "ML", tuned.predict_proba(X_val)[:, 1], tuned.predict_proba(X_test)[:, 1], tuned)
before, after = RESULTS["LightGBM"]["metrics"], RESULTS["LightGBM (Tuned)"]["metrics"]
print("Best parameters:", {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in best_params.items()}, f"| search time {(time.time() - t0) / 60:.1f} min")
d = after["PR_AUC"] - before["PR_AUC"]
interpret(f"Validation-month results of the best setting: PR-AUC {search.cv_results_['mean_test_pr_auc'][bi]:.4f}, recall {search.cv_results_['mean_test_recall'][bi]:.4f}, precision {search.cv_results_['mean_test_precision'][bi]:.4f}.",
          f"Test PR-AUC moved from {before['PR_AUC']:.4f} to {after['PR_AUC']:.4f} ({d:+.4f}); recall from {before['Recall']:.3f} to {after['Recall']:.3f}. Tuning " + ("helped." if d > 0.005 else "did not clearly help." if d > -0.005 else "slightly hurt on the test months."),
          "Tuning does not always help. A setting that is best on month 5 may not be best on months 6 and 7 because fraud patterns drift, and with few fraud cases small changes can be noise.")

### 4.1 ROC and Precision-Recall curves
The ROC curve is better when it hugs the top-left corner. The **Precision-Recall curve** is the stricter test for rare fraud: the dashed line shows what a random model would get (it equals the fraud rate). The huge number of genuine applications makes even a weak model look good on ROC, so the PR curve is the more honest of the two.

In [ ]:
def plot_curves(names, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    for n in names:
        f, t, _ = roc_curve(y_test, RESULTS[n]["test"]); ax[0].plot(f, t, label=f"{n} (AUC={RESULTS[n]['metrics']['ROC_AUC']:.3f})")
        p, r, _ = precision_recall_curve(y_test, RESULTS[n]["test"]); ax[1].plot(r, p, label=f"{n} (AP={RESULTS[n]['metrics']['PR_AUC']:.3f})")
    ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(title="ROC curves" + title, xlabel="False positive rate", ylabel="Recall"); ax[0].legend(fontsize=8)
    ax[1].axhline(y_test.mean(), c="k", ls="--"); ax[1].set(title="Precision-Recall curves" + title, xlabel="Recall", ylabel="Precision"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
plot_curves(list(RESULTS))
interpret("A curve that stays above the others over the whole range is the better ranker; crossing curves mean the best model depends on how many alerts the bank can handle.",
          f"The random-model line in the PR plot sits at {y_test.mean():.3f}. Curves far above it show real learning. Most of the difference between models appears at the left of the PR plot, where the very riskiest applications are.")

---
# Phase 5: Deep Learning Track (Keras Neural Network)

A neural network needs scaled, numeric input. Our preprocessing already gives that (numbers scaled, categories one-hot encoded, gaps filled), and the validation month is kept for early stopping, so no extra split is needed. The test months stay untouched.

**Architecture (chosen for this data: many rows but very few fraud cases and only a few dozen inputs, so a small and regularised network; the exact row, fraud and input counts are printed by the cells above and below).**

| Part | Choice | Reason |
|---|---|---|
| Input | One neuron per column | Takes the preprocessed features |
| Hidden layers | 128, 64, 32 neurons, ReLU | Learns feature combinations while narrowing step by step; ReLU trains fast |
| Batch normalisation | After every hidden layer | Stabilises training with mixed binary and continuous inputs |
| Dropout | 0.3, 0.3, 0.2 | Randomly switches neurons off so the network cannot memorise |
| Output | 1 neuron, sigmoid | Fraud probability between 0 and 1 |
| Loss / optimizer | Binary cross-entropy with class weights / Adam (lr 0.001) | Standard yes-no loss; the fraud weight is the square root of the imbalance ratio (the full ratio makes training noisy) |
| Batch size / epochs | 1024 / up to `EPOCHS` | Large batches make sure every batch contains some fraud |
| Callbacks | EarlyStopping on validation PR-AUC, ReduceLROnPlateau | Stop when learning stops helping; lower the learning rate when progress stalls |

In [ ]:
def build_mlp(input_dim):
    inp = keras.Input(shape=(input_dim,)); x = inp
    for units, drop in [(128, 0.3), (64, 0.3), (32, 0.2)]:
        x = layers.Dense(units, kernel_regularizer=regularizers.l2(1e-5))(x)
        x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(drop)(x)
    model = keras.Model(inp, layers.Dense(1, activation="sigmoid")(x), name="fraud_mlp")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=[keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall"),
                           keras.metrics.AUC(name="roc_auc"), keras.metrics.AUC(curve="PR", name="pr_auc")])
    return model

keras.utils.set_random_seed(SEED)
mlp = build_mlp(X_train.shape[1])
class_weight = {0: 1.0, 1: float(np.sqrt(SPW))}
print("Class weights:", class_weight, "| fraud cases in train/validation/test:", int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))
mlp.summary()

In [ ]:
cb = [callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=8, restore_best_weights=True, verbose=1),
      callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=3, min_lr=1e-5, verbose=1)]
history = mlp.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=1024, class_weight=class_weight, callbacks=cb, verbose=2)
hist = pd.DataFrame(history.history); print(f"Stopped after {len(hist)} epochs")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (m, t) in zip(ax, [("loss", "Loss"), ("pr_auc", "PR-AUC"), ("recall", "Recall (threshold 0.5)")]):
    a.plot(hist[m], label="Training", c=BLUE); a.plot(hist[f"val_{m}"], label="Validation", c=RED); a.set(title=t, xlabel="Epoch"); a.legend()
plt.tight_layout(); plt.show()

dl_val = mlp.predict(X_val, batch_size=8192, verbose=0).ravel(); dl_test = mlp.predict(X_test, batch_size=8192, verbose=0).ravel()
DL_NAME = "Deep Learning (MLP)"; add_result(DL_NAME, "DL", dl_val, dl_test, mlp)
be = int(hist["val_pr_auc"].idxmax())
interpret(f"Best epoch by validation PR-AUC: {be + 1} (validation {hist['val_pr_auc'].max():.3f}, training {hist['pr_auc'].iloc[be]:.3f}, gap {hist['pr_auc'].iloc[be] - hist['val_pr_auc'].iloc[be]:+.3f}). At the last epoch, validation loss minus training loss is {hist['val_loss'].iloc[-1] - hist['loss'].iloc[-1]:+.3f}.",
          "**Overfitting check:** overfitting shows as training metrics that keep improving while validation metrics stall or get worse. Early stopping with restored best weights, dropout and L2 limit it. Part of the gap is also expected because the validation month is *later* than the training months (fraud drifts), and dropout is active only during training.")

---
# Phase 6: Evaluation, Model Selection and Threshold

**Reading the confusion matrix here.** TP = fraud caught. TN = genuine application approved. FP = genuine customer wrongly flagged (delay, friction, analyst time). FN = fraud missed (direct financial loss). **False negatives are usually the costlier error**, so we pay close attention to recall while keeping the alert volume manageable.

We now put all models side by side. The **final model** is the one with the highest **PR-AUC** (the best threshold-free measure for rare fraud). We also show which model is cheapest under the assumed costs `COST_FN` and `COST_FP`, which are illustrative, not real bank numbers.

In [ ]:
comparison = pd.DataFrame({n: r["metrics"] for n, r in RESULTS.items()}).T.astype(float)
comparison.insert(0, "Type", [r["kind"] for r in RESULTS.values()])
comparison["Expected_cost"] = comparison["FN"] * COST_FN + comparison["FP"] * COST_FP
NO_MODEL_COST = float(y_test.sum() * COST_FN)               # approve everybody: every fraud gets through
comparison = comparison.sort_values("PR_AUC", ascending=False)
show_cols = ["Type", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TPR@5%FPR", "TP", "FP", "FN", "Expected_cost"]
display(comparison[show_cols].style.format({c: "{:.4f}" for c in ["Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TPR@5%FPR"]} | {"Expected_cost": "{:,.0f}"})
        .background_gradient(subset=["Recall", "F1", "PR_AUC"], cmap="Greens"))

names = list(comparison.index)
fig, axes = plt.subplots(1, len(names), figsize=(4.1 * len(names), 3.8))
for a, n in zip(np.atleast_1d(axes), names):
    m = RESULTS[n]["metrics"]; sns.heatmap([[m["TN"], m["FP"]], [m["FN"], m["TP"]]], annot=True, fmt=".0f", cmap="Blues", cbar=False, ax=a,
                                           xticklabels=["Pred genuine", "Pred fraud"], yticklabels=["Actual genuine", "Actual fraud"]); a.set_title(f"{n}\nRecall={m['Recall']:.2f}", fontsize=9)
plt.tight_layout(); plt.show()
plot_curves([DL_NAME] + [n for n in names if RESULTS[n]["kind"] == "ML"][:2], " (MLP vs top ML models)")

best_ml_name = comparison[comparison["Type"] == "ML"]["PR_AUC"].idxmax()
FINAL_NAME = comparison["PR_AUC"].idxmax()
best_business = comparison.sort_values(["Expected_cost", "PR_AUC"], ascending=[True, False]).index[0]
fm, bm, dm = comparison.loc[FINAL_NAME], comparison.loc[best_ml_name], comparison.loc[DL_NAME]
interpret(f"**Final model: {FINAL_NAME}** (highest PR-AUC {fm['PR_AUC']:.4f}); at its validation-chosen threshold it has recall {fm['Recall']:.3f}, precision {fm['Precision']:.3f}, F1 {fm['F1']:.3f}.",
          f"Best recall: {comparison['Recall'].idxmax()} ({comparison['Recall'].max():.3f}). Best precision: {comparison['Precision'].idxmax()} ({comparison['Precision'].max():.3f}). Best F1: {comparison['F1'].idxmax()} ({comparison['F1'].max():.3f}). Cheapest under the assumed costs at these thresholds: **{best_business}**.",
          f"Best ML model: {best_ml_name} (PR-AUC {bm['PR_AUC']:.4f}). Neural network: PR-AUC {dm['PR_AUC']:.4f}, which " + ("beat" if dm["PR_AUC"] > bm["PR_AUC"] else "did **not** beat") + " the best ML model. On table data like this, boosted trees are often as good as or better than a neural network, so either result is normal.",
          "Close scores can swap places on another split because the test months have few fraud cases. Do not over-read tiny gaps (about 0.01 PR-AUC or less).")

### 6.1 Threshold optimisation
A model outputs a score, and the threshold decides when it becomes a "fraud" flag. A **lower threshold** catches more fraud (higher recall) but creates more false alarms; a **higher threshold** gives fewer false alarms but lets more fraud through. We choose the threshold on the **validation month**, then apply it once to the test months. Choosing it on the test months would make the result look better than it is.

In [ ]:
s_val, s_test = RESULTS[FINAL_NAME]["val"], RESULTS[FINAL_NAME]["test"]

def sweep(yt, p):
    rows = []
    for t in np.unique(np.quantile(p, np.linspace(0.5, 0.9999, 300))):       # candidate thresholds from the score distribution
        tn, fp, fn, tp = confusion(yt, p, t)
        pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        rows.append({"threshold": t, "precision": pr, "recall": rc, "f1": 2 * pr * rc / max(pr + rc, 1e-12), "cost": fn * COST_FN + fp * COST_FP})
    return pd.DataFrame(rows)
sw = sweep(y_val, s_val)
thr_f1 = float(sw.loc[sw["f1"].idxmax(), "threshold"])
thr_cost = float(np.median(sw.loc[sw["cost"] == sw["cost"].min(), "threshold"]))
thr_fpr5, thr_fpr1 = thr_at_fpr(y_val, s_val, 0.05), thr_at_fpr(y_val, s_val, 0.01)
RECOMMENDED_THR = thr_cost

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(sw["threshold"], sw["precision"], label="Precision", c=BLUE); ax[0].plot(sw["threshold"], sw["recall"], label="Recall", c=RED); ax[0].plot(sw["threshold"], sw["f1"], "g--", label="F1")
for t, l in [(thr_f1, "max F1"), (thr_cost, "min cost")]: ax[0].axvline(t, c="grey", ls=":"); ax[0].text(t, 0.02, l, rotation=90, fontsize=8)
ax[0].set(title="Precision, recall, F1 vs threshold\n(validation month)", xlabel="Threshold"); ax[0].legend()
ax[1].plot(sw["threshold"], sw["cost"] / len(y_val) * 10000, c="black"); ax[1].axvline(thr_cost, c="grey", ls=":")
ax[1].set(title=f"Assumed cost per 10,000 applications (missed fraud={COST_FN}, false alarm={COST_FP})", xlabel="Threshold")
plt.tight_layout(); plt.show()

ops = {"Default 0.5": 0.5, "Max F1": thr_f1, "Lowest assumed cost": thr_cost, "5% false-alarm budget": thr_fpr5, "1% false-alarm budget": thr_fpr1}
tt = pd.DataFrame({lbl: {**compute_metrics(y_test, s_test, t), "Threshold": t} for lbl, t in ops.items()}).T.astype(float)
tt["Alerts per 10k"] = (tt["TP"] + tt["FP"]) / len(y_test) * 10000; tt["Total cost"] = tt["FN"] * COST_FN + tt["FP"] * COST_FP
display(tt[["Threshold", "Precision", "Recall", "F1", "TP", "FP", "FN", "Alerts per 10k", "Total cost"]].round(4))
rc_, d5 = tt.loc["Lowest assumed cost"], tt.loc["Default 0.5"]
interpret(f"Thresholds chosen on the validation month: max F1 {thr_f1:.3f}, lowest assumed cost {thr_cost:.3f}, 5% false-alarm budget {thr_fpr5:.3f}, 1% budget {thr_fpr1:.3f}. **Recommended: {RECOMMENDED_THR:.3f}** (lowest assumed cost).",
          f"On the test months it gives recall {rc_['Recall']:.3f} and precision {rc_['Precision']:.3f}, against recall {d5['Recall']:.3f} and precision {d5['Precision']:.3f} at 0.5, with about {rc_['Alerts per 10k']:.0f} alerts per 10,000 applications.",
          "**Maximum fraud catch vs fewer false alarms:** a low threshold catches the most fraud but floods the analysts; a high one keeps the queue short but lets fraud through. The best point depends on the real cost ratio and on how many alerts the team can review, which is why the costs at the top are editable.")

---
# Phase 7: Explainability and Business Interpretation

A fraud team must be able to explain **why** an application was flagged. For the best ML model we use two views: **built-in feature importance** and **permutation importance** (shuffle one column in the test data and measure how much PR-AUC drops, which works for any model). A third view, **SHAP**, shows the direction and size of each feature's effect; it is skipped safely if it fails in your environment.

In [ ]:
est = RESULTS[best_ml_name]["estimator"]
native = pd.Series(est.feature_importances_ if hasattr(est, "feature_importances_") else np.abs(est.coef_.ravel()), index=FEAT_NAMES)
native = (native / native.sum()).sort_values(ascending=False)
idx = np.random.RandomState(SEED).choice(len(y_test), size=min(40000, len(y_test)), replace=False)
perm = permutation_importance(est, X_test[idx], y_test[idx], scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=1)
perm_df = pd.Series(perm.importances_mean, index=FEAT_NAMES).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
perm_df.head(12)[::-1].plot(kind="barh", color=RED, ax=ax[0]); ax[0].set_title(f"{best_ml_name}: permutation importance (drop in PR-AUC)")
native.head(12)[::-1].plot(kind="barh", color=BLUE, ax=ax[1]); ax[1].set_title("Built-in importance (share of total)")
plt.tight_layout(); plt.show()

MEANING = {"name_email_similarity": "Email unlike the applicant's name (possible fake identity)", "email_is_free": "Free email addresses are cheap to create in bulk",
           "velocity_6h": "Burst of applications in 6 hours (automated attack)", "velocity_24h": "Application burst over 24 hours", "velocity_4w": "Application volume over 4 weeks",
           "velocity_6h_to_24h": "Acceleration of application bursts", "velocity_24h_to_4w": "Acceleration compared with the 4-week level",
           "zip_count_4w": "Many applications from one area recently", "bank_branch_count_8w": "Unusual concentration at one branch",
           "date_of_birth_distinct_emails_4w": "Same birth date used with many emails (identity reuse)", "device_distinct_emails_8w": "One device used with many emails",
           "credit_risk_score": "The bank's own risk score of the applicant", "proposed_credit_limit": "High requested limit means higher possible loss",
           "credit_limit_to_income": "Limit that is high compared with income", "customer_age": "Age bucket (sensitive attribute, see fairness check)", "income": "Declared income (binned)",
           "days_since_request": "Days since the application was made", "session_length_in_minutes": "Very short sessions can mean scripts or bots",
           "keep_alive_session": "'Keep me logged in' behaviour", "foreign_request": "Request from another country", "has_other_cards": "Existing relationship with the bank",
           "phone_home_valid": "Home phone could be verified", "phone_mobile_valid": "Mobile phone could be verified", "phones_valid": "Number of verified phones",
           "current_address_months_count": "Time at current address", "prev_address_months_count": "Time at previous address", "bank_months_count": "Age of previous bank account",
           "intended_balcon_amount": "Initial transfer amount", "n_missing_flags": "How many history fields are missing", "address_months_total": "Total months across addresses",
           "address_to_age": "Time at address compared with age", "low_similarity_free_email": "Free email AND email unlike the name"}
def meaning(f):
    if f in MEANING: return MEANING[f]
    if f.endswith("_missing"): return f"History field '{f[:-8]}' is missing"
    for pre in CAT_RAW:
        if f.startswith(pre + "_"): return f"Category '{f[len(pre) + 1:]}' of {pre} (anonymised level)"
    return ""
top_perm_names, native_top = perm_df.head(5).index.tolist(), native.head(6).index.tolist()
tab = perm_df.head(8).to_frame("PR-AUC drop when shuffled"); tab["What it means"] = [meaning(f) for f in tab.index]; display(tab)
interpret(f"Most important columns by permutation: **{', '.join(top_perm_names)}**. Shuffling `{perm_df.index[0]}` lowers PR-AUC by {perm_df.iloc[0]:.4f}, more than any other column.",
          f"Top built-in features: {', '.join(native_top)}.",
          f"Engineered features in the permutation top 8: {[f for f in perm_df.head(8).index if f in NEW_FEATS] or 'none'}.",
          "Values near zero mean the model barely uses that column. Importance shows association, not cause, and fraudsters change their behaviour over time.")

In [ ]:
# SHAP: global summary plus one local explanation (the highest-scoring truly fraudulent test application)
try:
    import shap
    fraud_idx = np.where(y_test == 1)[0]; top_local = int(fraud_idx[np.argmax(RESULTS[best_ml_name]["test"][fraud_idx])])
    rows = sorted(set(np.random.RandomState(SEED).choice(len(X_test), min(1500, len(X_test)), replace=False).tolist() + [top_local]))
    Xs = X_test[rows]
    expl = shap.TreeExplainer(est); sv = expl.shap_values(Xs)
    sv = sv[1] if isinstance(sv, list) else np.asarray(sv); sv = sv[:, :, 1] if sv.ndim == 3 else sv
    shap.summary_plot(sv, Xs, feature_names=FEAT_NAMES, max_display=15, show=False); plt.title("SHAP summary"); plt.tight_layout(); plt.show()
    i = rows.index(top_local); ev = np.ravel(expl.expected_value)[-1]
    shap.plots.waterfall(shap.Explanation(values=sv[i], base_values=ev, data=Xs[i], feature_names=list(FEAT_NAMES)), max_display=10, show=False); plt.tight_layout(); plt.show()
    interpret("In the summary, each dot is one application; red = high feature value, blue = low; dots on the right push the prediction towards **fraud**.",
              "The waterfall explains one confirmed fraud application: bars to the right raised its fraud score, bars to the left lowered it. An analyst can read this as the reason for the flag.")
except Exception as err:
    print("SHAP was skipped:", repr(err), "\nThe permutation importance above is already a valid explanation.")

### 7.1 What this means for the bank
**What the model predicts:** a fraud risk score for every new account application. **How a bank would use it:** applications with a low score are approved automatically; applications above the threshold go to an analyst queue **sorted from the highest score down**, and the very highest scores trigger an extra check (ID document, call-back) before the account is opened; confirmed fraud outcomes are saved and used to retrain the model. **False positives** delay a genuine customer and cost analyst time. **False negatives** let a fraudulent account open, which can mean unpaid credit, laundered money and fines, so they are usually the more expensive mistake. The model does **not replace** analysts: it focuses their limited time, and the explanations above tell them why an application looks risky.

In [ ]:
final_prob = RESULTS[FINAL_NAME]["test"]; test_df = df_p.loc[te].reset_index(drop=True); n_test = len(y_test)
mt = compute_metrics(y_test, final_prob, RECOMMENDED_THR)
high_cut = max(thr_fpr1, RECOMMENDED_THR + 1e-6)
band = pd.cut(final_prob, [-0.01, RECOMMENDED_THR, high_cut, 1.01], labels=["Low (approve)", "Medium (analyst review)", "High (extra check first)"])
bt = pd.DataFrame({"band": band, "actual": y_test}).groupby("band", observed=False)["actual"].agg(applications="size", actual_fraud="sum")
bt["fraud_rate_%"] = (bt["actual_fraud"] / bt["applications"].replace(0, np.nan) * 100).round(2); display(bt)

order = np.argsort(-final_prob)
view = test_df.loc[order[:10], ["customer_age", "income", "velocity_6h", "credit_risk_score", "name_email_similarity", "device_distinct_emails_8w"]].assign(
    fraud_score=final_prob[order[:10]], actual=pd.Series(y_test[order[:10]]).map({0: "Genuine", 1: "Fraud"}).values)
print("TOP 10 HIGHEST-RISK TEST APPLICATIONS"); display(view)
cum = np.cumsum(y_test[order]) / max(y_test.sum(), 1)
gains = {k: float(cum[int(k * n_test) - 1]) for k in (0.01, 0.05, 0.10)}
interpret(f"At threshold {RECOMMENDED_THR:.3f} the model sends **{(mt['TP'] + mt['FP']) / n_test * 10000:.0f} of every 10,000 applications** to review. It catches **{int(mt['TP'])} of {int(mt['TP'] + mt['FN'])}** frauds in the test months, misses **{int(mt['FN'])}** and raises **{int(mt['FP'])}** false alarms.",
          f"An analyst checks about {(mt['TP'] + mt['FP']) / max(mt['TP'], 1):.1f} flagged applications to find one real fraud. Reviewing only the riskiest 1% of applications finds {gains[0.01]:.1%} of all fraud, the riskiest 5% finds {gains[0.05]:.1%}, the riskiest 10% finds {gains[0.10]:.1%}.",
          f"Under the assumed costs, the total cost drops from {NO_MODEL_COST:,.0f} (no model) to {mt['FN'] * COST_FN + mt['FP'] * COST_FP:,.0f}, a reduction of {(1 - (mt['FN'] * COST_FN + mt['FP'] * COST_FP) / NO_MODEL_COST):.1%}. This depends entirely on the assumed unit costs.",
          "Scores are not exact percentages because class weights were used; use them for **ranking**.")

### 7.2 Fairness check by age group
`customer_age` is a sensitive attribute. We apply the **same recommended threshold** to applicants under 50 and 50 or older and compare error rates. A false-alarm rate that is very different between groups means genuine customers of one group get delayed more often. We only *measure* the gap here; choosing a fix is a policy decision.

In [ ]:
group = np.where(test_df["customer_age"].values >= 50, "age >= 50", "age < 50"); pred_final = (final_prob >= RECOMMENDED_THR).astype(int); rows = []
for g in ["age < 50", "age >= 50"]:
    m = group == g; tn_, fp_, fn_, tp_ = confusion_matrix(y_test[m], pred_final[m], labels=[0, 1]).ravel()
    rows.append({"group": g, "applications": int(m.sum()), "fraud_rate_%": y_test[m].mean() * 100, "flagged_%": pred_final[m].mean() * 100,
                 "recall": tp_ / max(tp_ + fn_, 1), "false_alarm_rate": fp_ / max(fp_ + tn_, 1), "precision": tp_ / max(tp_ + fp_, 1)})
fair = pd.DataFrame(rows).set_index("group"); display(fair)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, c in zip(ax, ["recall", "false_alarm_rate", "precision"]): sns.barplot(x=fair.index, y=fair[c], palette=[BLUE, RED], ax=a); a.set_title(c)
plt.tight_layout(); plt.show()
fpr_ratio = fair.loc["age >= 50", "false_alarm_rate"] / max(fair.loc["age < 50", "false_alarm_rate"], 1e-12)
interpret(f"False-alarm rate: {fair.loc['age < 50', 'false_alarm_rate']:.2%} for under 50 vs {fair.loc['age >= 50', 'false_alarm_rate']:.2%} for 50 and over. Ratio (50+ / under 50): **{fpr_ratio:.2f}** (1.00 would mean equal treatment).",
          f"Fraud rates also differ between the groups ({fair.loc['age < 50', 'fraud_rate_%']:.2f}% vs {fair.loc['age >= 50', 'fraud_rate_%']:.2f}%), so some difference in flag rates is expected. A false-alarm ratio far from 1 is the real fairness concern.")

---
# Conclusion

In [ ]:
f05 = RESULTS[FINAL_NAME]["metrics"]; fth = compute_metrics(y_test, final_prob, RECOMMENDED_THR)
dl_m, ml_m = RESULTS[DL_NAME]["metrics"], RESULTS[best_ml_name]["metrics"]
display(Markdown(f"""
**Problem.** Detect fraudulent bank-account-opening applications, a rare but costly kind of fraud, so the bank can review the riskiest applications first.

**Dataset.** The file had {N_RAW:,} applications; we used a {SAMPLE_FRAC:.0%} sample and after cleaning modelled {len(df_model):,} applications, **{n_pos_all:,} ({fraud_rate * 100:.2f}%) fraud**, about {imbalance_ratio:.0f} genuine per fraud. Always predicting 'genuine' would be {BASELINE_ACC * 100:.2f}% accurate and catch nothing. Data were split by time: train months 0-4, validation month 5, test months 6-7.

**EDA findings.** Largest missing-field gap: {gap.iloc[0]['column']} (fraud {gap.iloc[0]['fraud_if_missing']:.2f}% if missing vs {gap.iloc[0]['fraud_if_present']:.2f}% if present). Strongest linear correlation with fraud: {tcorr.index[0]} ({tcorr.iloc[0]:.3f}).

**Important features.** Permutation importance: {', '.join(top_perm_names)}. Built-in importance: {', '.join(native_top[:4])}.

**Models and imbalance.** Logistic Regression, Random Forest, LightGBM (baseline and tuned) and a Keras neural network. Imbalance handled mainly with class weights; in Phase 3, {imb_note}.

**Best ML model.** {best_ml_name}: PR-AUC {ml_m['PR_AUC']:.4f}, recall {ml_m['Recall']:.4f}, precision {ml_m['Precision']:.4f}, F1 {ml_m['F1']:.4f}. **Deep learning:** PR-AUC {dl_m['PR_AUC']:.4f}, recall {dl_m['Recall']:.4f}, precision {dl_m['Precision']:.4f}, which was {"better" if dl_m['PR_AUC'] > ml_m['PR_AUC'] else "not better"} than the best ML model.

**Final model: {FINAL_NAME}** (highest PR-AUC). ROC-AUC {f05['ROC_AUC']:.4f}, PR-AUC {f05['PR_AUC']:.4f}. At the recommended threshold **{RECOMMENDED_THR:.3f}**: recall {fth['Recall']:.4f}, precision {fth['Precision']:.4f}, catching {int(fth['TP'])} of {int(fth['TP'] + fth['FN'])} frauds with {int(fth['FP'])} false alarms.

**Business impact.** Analysts check about {(fth['TP'] + fth['FP']) / len(y_test) * 10000:.0f} applications per 10,000 instead of everything, while {int(fth['FN'])} of {int(fth['TP'] + fth['FN'])} frauds in the test months are missed. Fairness: the false-alarm ratio between age groups is {fpr_ratio:.2f}.
"""))

### Limitations
- **Few fraud cases in the test months:** metrics such as recall move a lot with a handful of cases, so all results carry uncertainty. A single time split gives no confidence interval.
- **Sample, not the full file:** we used a share of the 1,000,000 rows (`SAMPLE_FRAC`) to keep the run short. Using `1.0` would use all data but needs more memory and time.
- **Drift:** fraudsters adapt. The model is tested on later months, which is realistic, but it still needs monitoring and regular retraining.
- **Assumed costs:** the business costs used for the threshold are illustrative, not real bank figures.
- **Synthetic, anonymised data:** the BAF data were generated from a real dataset with privacy protection, so real-world performance must be checked again on real traffic. Many category names are anonymised.
- **Fairness:** only age was checked, and no fix was applied.
- **Limited tuning:** only a small random search was run; a neural network with more tuning could change the ML vs DL result.

### Future scope
Rolling (walk-forward) validation with confidence intervals, the other BAF variant files to test robustness to data shift, more boosting models (XGBoost, CatBoost) and ensembling with the MLP, probability calibration, focal loss, features that link shared devices or emails, fairness-aware training, a real-time API with an analyst feedback loop, and drift monitoring.

---
# Interview / Viva Preparation
The answers below are built from **your own run**, so the numbers match your notebook. Read them once after running everything, then practise saying them in your own words.

In [ ]:
gain = comparison.loc["LightGBM (Tuned)", "PR_AUC"] - comparison.loc["LightGBM", "PR_AUC"]
vs = "did not beat" if dl_m["PR_AUC"] <= ml_m["PR_AUC"] else "beat"
alerts = (fth["TP"] + fth["FP"]) / len(y_test) * 10000
qa = [
 ("1. Explain your fraud detection project.", f"I built a system that gives every bank account application a fraud risk score. Using the BAF dataset ({len(df_model):,} applications in my sample, {fraud_rate * 100:.2f}% fraud), I cleaned the data, handled the hidden missing values, engineered features, split the data by time, handled the imbalance, compared Logistic Regression, Random Forest, tuned LightGBM and a neural network, chose a threshold using business costs, explained the predictions and checked fairness by age."),
 ("2. Why fraud detection?", "Fraud is rare but costly, so it teaches imbalance, the right metrics, leakage, drift and business cost. A fraudulent account can mean unpaid credit and laundered money, while wrongly blocking a genuine customer also has a cost."),
 ("3. What is the target variable?", f"`fraud_bool` (1 = fraud, 0 = genuine): {n_pos_all:,} fraud and {n_neg_all:,} genuine applications."),
 ("4. What were the hidden missing values?", f"The file stores 'missing' as -1 (and negative numbers in `intended_balcon_amount`), not as an empty cell. The most affected column is {miss.index[0]} ({miss['hidden_missing_pct'].iloc[0]:.1f}%). I converted them to NaN, filled them with the training median and added a missing flag, because 'no history' can itself signal fraud."),
 ("5. Why a time split instead of a random split?", "Fraud changes month by month. A random split would mix months and make the test look easier than reality. I trained on months 0-4, tuned on month 5 and tested on months 6-7, so the model is tested on the future, like in a real bank."),
 ("6. Why is `month` not a feature?", "It is only used to split the data. If it were a feature, the model would learn the calendar instead of fraud behaviour, and the test months have values it never saw in training."),
 ("7. Why is accuracy not enough?", f"Because fraud is rare: always predicting 'genuine' gets {BASELINE_ACC * 100:.2f}% accuracy on my test months and catches no fraud. So I use precision, recall, F1, ROC-AUC and PR-AUC."),
 ("8. Explain precision and recall.", f"Precision is the share of flagged applications that are really fraud; recall is the share of all fraud that we catch. At the recommended threshold my final model has precision {fth['Precision']:.3f} and recall {fth['Recall']:.3f}."),
 ("9. Why is recall important, and what is the cost of precision?", f"A missed fraud can cost far more than a false alarm (I assumed {COST_FN} vs {COST_FP}). But low precision floods analysts: about {alerts:.0f} of every 10,000 applications are flagged at my threshold. The threshold balances both."),
 ("10. ROC-AUC vs PR-AUC?", f"ROC-AUC measures how well the model ranks fraud above genuine applications. Mine is {f05['ROC_AUC']:.3f}. PR-AUC ({f05['PR_AUC']:.3f}) is stricter for rare classes, because it ignores the huge number of easy genuine cases; a random model would only get about {y_test.mean():.3f}."),
 ("11. How did you handle imbalance?", f"I tested no handling, class weights, undersampling, oversampling and SMOTE, always on training data only. For the main models I used class weights: {imb_note}."),
 ("12. Why did you select your final model?", f"{FINAL_NAME} had the highest PR-AUC ({f05['PR_AUC']:.3f}), the best threshold-free measure for imbalanced data. I then chose the threshold separately, on the validation month, using business cost."),
 ("13. Why compare ML and DL?", f"To check whether a neural network adds value. It {vs} the best ML model ({best_ml_name}): PR-AUC {dl_m['PR_AUC']:.3f} vs {ml_m['PR_AUC']:.3f}. On table data like this, boosted trees are often as good and easier to explain."),
 ("14. What is overfitting and how did you control it?", f"Overfitting is memorising the training data and doing worse on new data. I used separate validation and test months, regularisation (dropout, batch normalisation, L2), early stopping (best epoch {be + 1}) and depth limits on the trees."),
 ("15. What is data leakage and how did you avoid it?", "Information from outside the training data reaching the model so scores look better than reality. I avoided it by splitting by time, fitting the median, scaler and encoder on the training months only, resampling only the training data, using row-by-row features, and choosing thresholds on the validation month, not on the test months."),
 ("16. How did you choose the threshold?", f"On the validation month. I compared max F1, lowest assumed cost and fixed false-alarm budgets, and recommended {RECOMMENDED_THR:.3f} (lowest cost). At that point the final model misses {int(fth['FN'])} of {int(fth['TP'] + fth['FN'])} frauds in the test months."),
 ("17. What are the most important features?", f"By permutation importance on {best_ml_name}: {', '.join(top_perm_names[:4])}. They show what an analyst should look at first."),
 ("18. How would a bank use your model?", f"Score each new application; about {alerts:.0f} per 10,000 go to review, highest risk first. Analysts confirm fraud, and their decisions feed retraining."),
 ("19. Did your model treat age groups fairly?", f"I compared false-alarm rates at the same threshold: ratio of {fpr_ratio:.2f} for 50+ versus under 50 (1.00 = equal). I only measured the gap; fixing it would be a policy decision."),
 ("20. Did tuning help, and what about changing fraud patterns?", f"Test PR-AUC for LightGBM changed by {gain:+.4f} after tuning, which is small enough to be partly noise. If fraud patterns change (drift), accuracy slowly drops, so I would monitor the flagged rate and precision, and retrain on newly reviewed applications."),
]
for q, a in qa: display(Markdown(f"**{q}**\n\n{a}\n"))